# Домашнее задание: Градиентный спуск для множественной регрессии

**Цель:** В этом домашнем задании вы самостоятельно реализуете полный цикл градиентного спуска для задачи множественной линейной регрессии, а затем проанализируете влияние гиперпараметров на его работу.

**Общая оценка за ДЗ: 10 баллов.**


## 0. Подготовка данных и Теория (0 баллов)

Сначала сгенерируем данные. Мы будем предсказывать значение `y`, основываясь на двух признаках (`X1`, `X2`). Обратите внимание, что масштабы признаков сильно отличаются — это важная деталь для одного из заданий.

**Наша модель:**
$$ \hat{y} = w_0 + w_1x_1 + w_2x_2 = \mathbf{w}^T \mathbf{x_b} $$

**Функция потерь (MSE):**
$$ L(\mathbf{w}) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2 $$

**Градиент функции потерь (в векторной форме):**
$$ \nabla L(\mathbf{w}) = \frac{2}{m} \mathbf{X_b}^T (\mathbf{X_b w} - \mathbf{y}) $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Генерируем данные
np.random.seed(42)
m = 100 # количество примеров
X1 = 100 * np.random.rand(m, 1)      # Признак 1 в большом масштабе
X2 = np.random.rand(m, 1)           # Признак 2 в малом масштабе
X = np.c_[X1, X2]                   # Объединяем признаки в матрицу

# Истинная зависимость: y = 5 + 3*x1 + 50*x2 + шум
y = 5 + 3 * X1 + 50 * X2 + np.random.randn(m, 1)
y = y.flatten() # Упрощаем до 1D-массива

# Добавляем фиктивный признак x0=1 для учета свободного члена w0 (bias)
X_b = np.c_[np.ones((m, 1)), X]

print("Размер матрицы признаков X_b:", X_b.shape)
print("Размер вектора y:", y.shape)

## Задание 1. Реализация градиентного спуска (4 балла)

Вам нужно написать две функции: одна будет вычислять градиент, а вторая — выполнять сам цикл спуска.

### 1.1. Функция для вычисления градиента (2 балла)

Реализуйте функцию, которая вычисляет вектор градиентов по формуле, приведённой выше.

In [ ]:
def calculate_gradient(X_b, y, w):
    """
    Вычисляет градиент MSE для множественной линейной регрессии.
    
    Args:
        X_b (np.array): Матрица признаков с добавленным столбцом единиц (m, n+1).
        y (np.array): Вектор истинных значений (m,).
        w (np.array): Вектор весов (n+1,).
        
    Returns:
        np.array: Вектор градиентов (n+1,).
    """
    m = len(y)
    
    # TODO: Реализуйте формулу градиента.
    # 1. Сделайте предсказания y_pred = X_b @ w
    # 2. Вычислите градиент по формуле: 2/m * X_b.T @ (y_pred - y)
    gradient = ...

    return gradient

In [ ]:
# Простой тест для вашей функции
w_test = np.zeros(X_b.shape[1])
grad_test = calculate_gradient(X_b, y, w_test)
assert grad_test.shape == (3,), "Размер градиента неверный!"
print("Тест на размерность градиента пройден!")

### 1.2. Функция градиентного спуска (2 балла)

Теперь реализуйте основной цикл. Функция должна инициализировать веса, а затем на каждой итерации вычислять градиент и обновлять веса. Также она должна сохранять историю значений функции потерь для последующей визуализации.

In [ ]:
def gradient_descent(X_b, y, learning_rate, num_iterations):
    """
    Выполняет градиентный спуск для поиска весов w.
    """
    m, n = X_b.shape
    # Инициализируем веса нулями
    w = np.zeros(n)
    
    # Хранилище для истории значений функции потерь
    loss_history = []
    
    # TODO: Реализуйте цикл градиентного спуска
    for i in range(num_iterations):
        # 1. Вычислить градиент с помощью вашей функции calculate_gradient()
        gradients = ...
        
        # 2. Обновить веса: w = w - learning_rate * gradients
        w = ...
        
        # 3. Рассчитать и сохранить текущее значение MSE
        y_pred = X_b @ w
        loss = np.mean((y_pred - y) ** 2)
        loss_history.append(loss)
        
    return w, loss_history

In [ ]:
# Запустим наш градиентный спуск с какими-нибудь параметрами
lr = 0.000001 # Эта скорость обучения подобрана для не-масштабированных данных
n_iter = 1000

final_w, loss_history = gradient_descent(X_b, y, lr, n_iter)

print("Финальные веса (w0, w1, w2):", final_w)
print("Последнее значение MSE:", loss_history[-1])

# Визуализируем, как менялась функция потерь
plt.figure(figsize=(10, 6))
plt.plot(range(n_iter), loss_history)
plt.title('Кривая обучения (Loss vs. Iterations)')
plt.xlabel('Итерация')
plt.ylabel('MSE')
plt.show()

## Задание 2. Эксперименты с гиперпараметрами (4 балла)

Правильный подбор гиперпараметров критически важен для градиентного спуска.

### 2.1. Влияние скорости обучения (Learning Rate) (2 балла)

Запустите ваш `gradient_descent` **три раза** с разной скоростью обучения (`learning_rate`):
1.  Слишком маленькой (например, `1e-7`).
2.  Оптимальной (используйте `1e-6` из примера выше).
3.  Слишком большой (например, `1e-5`).

Постройте на одном графике кривые обучения для всех трёх случаев и **письменно объясните** наблюдаемые результаты.

In [ ]:
# TODO: Запустите обучение 3 раза и соберите истории потерь
n_iter = 1000

learning_rates = {
    'Слишком маленькая (1e-7)': 1e-7,
    'Оптимальная (1e-6)': 1e-6,
    'Слишком большая (1e-5)': 1e-5
}

histories = {}
for name, lr in learning_rates.items():
    _, loss_hist = gradient_descent(X_b, y, lr, n_iter)
    histories[name] = loss_hist

# Визуализация
plt.figure(figsize=(12, 7))
for name, loss_hist in histories.items():
    plt.plot(range(n_iter), loss_hist, label=name)

plt.title('Сравнение скоростей обучения')
plt.xlabel('Итерация')
plt.ylabel('MSE')
plt.legend()
plt.ylim(0, 5000) # Ограничим ось y для наглядности
plt.show()

**Ваши выводы (объясните каждый из трёх случаев):**

1.  **Слишком маленькая LR:** (впишите сюда ваш ответ)
2.  **Оптимальная LR:** (впишите сюда ваш ответ)
3.  **Слишком большая LR:** (впишите сюда ваш ответ)


### 2.2. Влияние количества итераций (2 балла)

С помощью цикла `for` запустите обучение с оптимальной скоростью обучения (`1e-6`), но разным количеством итераций (например, 10, 50, 100, 500, 2000). Выведите на экран финальное значение MSE для каждого случая.

**Сделайте письменный вывод:** как количество итераций влияет на качество модели и в какой момент, по вашему мнению, обучение можно было бы остановить?

In [ ]:
# TODO: Проведите эксперимент с разным числом итераций
iteration_counts = [10, 50, 100, 500, 2000]
lr_optimal = 1e-6

for n_iter_test in iteration_counts:
    _, loss_hist_test = gradient_descent(X_b, y, lr_optimal, n_iter_test)
    print(f"Итераций: {n_iter_test}, Финальный MSE: {loss_hist_test[-1]:.2f}")


**Ваши выводы:**

(впишите сюда ваш ответ)

## Задание 3. Бонус: Масштабирование признаков (2 балла)

Как вы могли заметить, на исходных данных нам пришлось использовать очень маленькую скорость обучения, и сходимость была довольно медленной. Это происходит из-за того, что признаки `X1` и `X2` имеют очень разный масштаб. 

**Ваша задача:**

1.  Отмасштабируйте признаки `X` с помощью `StandardScaler` из `sklearn`.
2.  **Не забудьте** заново добавить к отмасштабированным данным столбец с единицами.
3.  Запустите градиентный спуск на новых, отмасштабированных данных. **Попробуйте подобрать более высокую скорость обучения** (например, начните с `0.1` или `0.01`).
4.  Сравните кривую обучения на масштабированных и не-масштабированных данных на одном графике.
5.  Сделайте вывод, почему масштабирование так сильно помогло.

In [ ]:
# 1. Масштабируем признаки
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Добавляем столбец с единицами
X_scaled_b = np.c_[np.ones((m, 1)), X_scaled]

# 3. Запускаем GD на масштабированных данных с новой, более высокой LR
# TODO: Подберите хорошую LR для масштабированных данных
lr_scaled = ... # Попробуйте 0.1, 0.01 и т.д.
n_iter_scaled = 1000
final_w_scaled, loss_history_scaled = gradient_descent(X_scaled_b, y, lr_scaled, n_iter_scaled)

print("Финальные веса на масштабированных данных:", final_w_scaled)
print("Последнее значение MSE на масштабированных данных:", loss_history_scaled[-1])

# 4. Сравнение кривых обучения
plt.figure(figsize=(12, 7))
plt.plot(range(len(loss_history)), loss_history, label='Не-масштабированные данные (lr=1e-6)')
plt.plot(range(len(loss_history_scaled)), loss_history_scaled, label=f'Масштабированные данные (lr={lr_scaled})')
plt.title('Сравнение сходимости на разных данных')
plt.xlabel('Итерация')
plt.ylabel('MSE')
plt.legend()
plt.yscale('log') # Логарифмическая шкала для наглядности
plt.show()

**Ваши выводы:**

(впишите сюда ваш ответ)